<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/03_heavy_tailed_slope_prior.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 3 — Heavy-tailed slope prior

Replace the tight Normal slope prior with a Student-t prior having the same central scale but heavier tails. This isolates the effect of prior tail behavior.

## Setup

This course pins PyMC, modular ArviZ, and Bambi for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1" \
    "bambi==0.21.0"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bambi as bmb
import pymc as pm
import arviz_base as azb
import arviz_stats as azs
import arviz_plots as azp

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("Bambi:", bmb.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

That zero point is scientifically meaningful, so every Bambi model in this sequence uses `center_predictors=False`. The `Intercept` prior is therefore a prior on baseline reaction time rather than reaction time at the average deprivation day.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

print(f"{sleep['Subject'].nunique()} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

# 3.1 Prior tail behavior

How does a heavy-tailed slope prior differ from a tight Normal prior before seeing the data?

$$\beta\sim t_7(0,1).$$

Near zero this resembles the previous $N(0,1)$ prior, but it leaves substantially more probability for larger effects.

In [ ]:
priors = {
    "Intercept": bmb.Prior("Normal", mu=250, sigma=100),
    "Days": bmb.Prior("StudentT", nu=7, mu=0, sigma=1),
    "sigma": bmb.Prior("Exponential", lam=0.02),
}
model = bmb.Model(
    "Reaction ~ Days", sleep, family="gaussian", priors=priors, center_predictors=False
)
model

In [ ]:
from scipy import stats

x = np.linspace(-4, 4, 500)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(x, stats.norm.pdf(x, 0, 1), label="Normal(0, 1)")
ax.plot(x, stats.t.pdf(x, df=7), label="Student-t(7, 0, 1)")
ax.set(xlabel="Daily effect (ms/day)", ylabel="Density")
ax.legend(frameon=False)
plt.show()

print("95% interval for Exponential(0.02):", stats.expon(scale=1/0.02).ppf([0.025, 0.975]))

In [ ]:
prior = model.prior_predictive(draws=500, random_seed=RANDOM_SEED)
azp.plot_ppc_dist(
    prior,
    group="prior_predictive",
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

# 3.2 Posterior impact

Does allowing rare large slopes change what the data say about the deprivation effect?

In [ ]:
idata = model.fit(
    draws=1000,
    tune=1500,
    chains=4,
    target_accept=0.90,
    random_seed=RANDOM_SEED,
)

print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))


In [ ]:
azs.summary(idata, var_names=["Intercept", "Days", "sigma"], ci_prob=0.90, ci_kind="hdi", round_to=2)

In [ ]:
azp.plot_trace_dist(idata, var_names=["Intercept", "Days", "sigma"]);

In [ ]:
bmb.interpret.plot_predictions(
    model,
    idata,
    conditional="Days",
    target="mean",
    prob=[0.50, 0.90],
)

# 3.3 Predictive impact

Does prior tail behavior matter enough here to change the model’s predictive story?

In [ ]:
model.predict(
    idata,
    kind="response",
    inplace=True,
    random_seed=RANDOM_SEED,
)

azp.plot_ppc_dist(
    idata,
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

In [ ]:
model.compute_log_likelihood(idata)
model.compute_log_prior(idata)

azs.psense_summary(idata, var_names=["Intercept", "Days", "sigma"])

azp.plot_psense_dist(
    idata,
    var_names=["Intercept", "Days", "sigma"],
    visuals={"dist": False},
);

# 3.4 Structural limitation

Is the main limitation the slope-prior family, or the absence of participant-level structure?